### why we need attention
1. sequential nature of seq2seq architectures precludes parallelisation within training examples. this becomes critical at longer `seq_len`. even with improvements, fundamental constraint remains.
2. conv based architectures (eg ByteNet and ConvS2S) have hard times learning dependencies between distant positions

In [14]:
import torch
from torch import nn
import torch.nn.functional as F
import requests
import os

### attention block
- this is multi-head self-attention

$\text{attention } = \text{softmax}(\frac{Q K ^ T}{\sqrt{d_k}})V$

input `x: (B, L, D)`

**QKV projections** — single matmul then split:
- `W_q, W_k, W_v` are all `nn.Linear(D, D)`
- produces `Q, K, V: (B, L, D)`
- split into heads by *only* splitting across last dim, then transposing:
```
  Q = Q.view(B, L, num_heads, head_dim).transpose(1, 2)  # (B, num_heads, L, head_dim)
```
- never do `.view(B, -1, L, head_dim)` directly — correct shape but scrambles values across L and D

**Scaled dot-product attention** — per head:
```
attn = softmax( Q @ K^T / sqrt(head_dim) )   # (B, num_heads, L, L)
out  = attn @ V                               # (B, num_heads, L, head_dim)
```

**Reassemble heads** — transpose back *before* reshape:
```
out = out.transpose(1, 2).contiguous().view(B, L, D)
```
- `.contiguous()` required because transpose only changes strides, not memory layout
- `view` requires contiguous memory

**Output projection** — `W_o: nn.Linear(D, D)`, mixes info across heads

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, embedding_size, num_heads):
        super().__init__()
        assert embedding_size % num_heads == 0
        self.num_heads = num_heads
        self.heads_dim = embedding_size // num_heads

        self.W_q = nn.Linear(embedding_size, embedding_size)
        self.W_k = nn.Linear(embedding_size, embedding_size)
        self.W_v = nn.Linear(embedding_size, embedding_size)
        self.W_o = nn.Linear(embedding_size, embedding_size)

    def forward(self, x):
        B, L, D = x.shape

        Q = self.W_q(x).view(B, L, self.num_heads, self.heads_dim).transpose(1, 2)
        K = self.W_k(x).view(B, L, self.num_heads, self.heads_dim).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.num_heads, self.heads_dim).transpose(1, 2)
        # (B, num_heads, L, heads_dim)

        # causal mask — upper triangle = -inf so future tokens are invisible
        QK_T = torch.matmul(Q, K.transpose(-1, -2)) / (self.heads_dim ** 0.5)
        mask = torch.triu(torch.ones(L, L, device=x.device), diagonal=1).bool()
        QK_T = QK_T.masked_fill(mask, float("-inf"))
        attn = torch.softmax(QK_T, dim=-1)
        # (B, num_heads, L, L)

        out  = torch.matmul(attn, V)
        # (B, num_heads, L, heads_dim)

        out = out.transpose(1, 2).contiguous().view(B, L, D)
        return self.W_o(out)

### transformer block

## Information Flow in Transformers

---

### V in Attention
- $QK^T$ determines *where to look* — the attention pattern
- $V$ determines *what to extract* from those positions
- $A_i V_i$ = for each position, take a weighted mixture of value vectors from attended positions
- result: each token's embedding now contains a blend of information from its context, weighted by relevance

---

### FFN as Key-Value Memory
FFN structure: $\text{GELU}(xW_1)W_2$

- $W_1$ expands to `hidden_dim` (typically $4D$) — acts as **pattern detector**
- $W_2$ projects back to $D$ — stores **what to output** when that pattern fires
- interpretation (Geva et al. 2021): individual neurons in $W_1$ are pattern matchers,
  $W_2$ encodes "if this pattern fires, add this concept to the stream"
- attention = *communication* between positions, FFN = *computation* within each position

---

### Residual Stream
every layer just adds a small update $\Delta$ to the original stream:

$$X^{(l)} = X^{(0)} + \sum_{k=1}^{l} \Delta_\text{attn}^{(k)} + \Delta_\text{ffn}^{(k)}$$

- nothing gets overwritten — early information persists untouched if later layers don't modify it
- each attention head and FFN block is just *writing* small updates to a shared stream
- a token that carries simple info (e.g. fullstop) may pass through most layers unchanged

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, hidden_dim):
        super().__init__()
        self.attn = AttentionBlock(embedding_dim, num_heads)
        self.ffn  = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embedding_dim),
        )
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        x = self.ln1(x + self.attn(x))
        x = self.ln2(x + self.ffn(x))
        return x

### LayerNorm
$y = \frac{x - E[x]}{\sqrt{Var[x] + \epsilon}} * \gamma + \beta$ 

where: 
- $E[x]$ - mean 
- $Var[x]$ - variance 
- $\epsilon$ - variable constant
- $\gamma \text{ and } \beta$ - learnable parameters 

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, normalized_shape, epsilon=1e-5):
        super().__init__()
        self.epsilon = epsilon
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta  = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        x    = (x - mean) / (var + self.epsilon).sqrt()
        return self.gamma * x + self.beta

### GPT architecture

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads, hidden_dim, context_len, num_transformer_blocks):
        super().__init__()
        self.context_len = context_len
        self.vocab_size = vocab_size

        self.tok_emb = nn.Embedding(vocab_size, embedding_dim)
        self.pos_emb = nn.Embedding(context_len, embedding_dim)
        self.blocks  = nn.Sequential(*[
            TransformerBlock(embedding_dim, num_heads, hidden_dim)
            for _ in range(num_transformer_blocks)
        ])
        self.ln_f  = nn.LayerNorm(embedding_dim)
        self.head  = nn.Linear(embedding_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, L = idx.shape
        positions = torch.arange(L, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(positions)  # (B, L, D)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (B, L, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1))

        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_crop = idx[:, -self.context_len:]
            logits, _ = self(idx_crop)
            probs = torch.softmax(logits[:, -1, :], dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_tok], dim=1)
        return idx

## examlpe training

### Duplicate/Overlapping Batches in Transformer Training

random sampling with `torch.randint` means batches can overlap or repeat — this is intentional:
- ~1M possible windows in TinyShakespeare (1MB text, context_len=64), so collision probability per step is low
- overlapping windows = same text seen with different context, acts as data augmentation
- SGD benefits from this noise rather than being hurt by it

alternative is a proper dataloader that iterates sequentially through the dataset without replacement,
but random sampling works well in practice for language modelling

In [63]:
# ---- hyperparams ----
EMBEDDING_DIM  = 128
NUM_HEADS      = 4
HIDDEN_DIM     = 512
NUM_LAYERS     = 4
CONTEXT_LEN    = 64
BATCH_SIZE     = 64
LR             = 3e-4
STEPS          = 5000
EVAL_INTERVAL  = 500
EVAL_STEPS     = 50
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

# ---- data ----
def get_data():
    path = "shakespeare.txt"
    if not os.path.exists(path):
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        print("Downloading TinyShakespeare...")
        r = requests.get(url)
        with open(path, "w") as f:
            f.write(r.text)
    with open(path, "r") as f:
        text = f.read()
    return text

text = get_data()
chars = sorted(set(text))
VOCAB_SIZE = len(chars)
print(f"Vocab size: {VOCAB_SIZE}, dataset size: {len(text):,} chars")

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - CONTEXT_LEN, (BATCH_SIZE,))
    x = torch.stack([d[i:i+CONTEXT_LEN] for i in ix])
    y = torch.stack([d[i+1:i+CONTEXT_LEN+1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

# ---- model ----
class AttentionBlock(nn.Module):
    def __init__(self, embedding_size, num_heads):
        super().__init__()
        assert embedding_size % num_heads == 0
        self.num_heads = num_heads
        self.heads_dim = embedding_size // num_heads

        self.W_q = nn.Linear(embedding_size, embedding_size)
        self.W_k = nn.Linear(embedding_size, embedding_size)
        self.W_v = nn.Linear(embedding_size, embedding_size)
        self.W_o = nn.Linear(embedding_size, embedding_size)

    def forward(self, x):
        B, L, D = x.shape

        Q = self.W_q(x).view(B, L, self.num_heads, self.heads_dim).transpose(1, 2)
        K = self.W_k(x).view(B, L, self.num_heads, self.heads_dim).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.num_heads, self.heads_dim).transpose(1, 2)
        # (B, num_heads, L, heads_dim)

        # causal mask — upper triangle = -inf so future tokens are invisible
        QK_T = torch.matmul(Q, K.transpose(-1, -2)) / (self.heads_dim ** 0.5)
        mask = torch.triu(torch.ones(L, L, device=x.device), diagonal=1).bool()
        QK_T = QK_T.masked_fill(mask, float("-inf"))

        attn = torch.softmax(QK_T, dim=-1)
        out  = torch.matmul(attn, V)
        # (B, num_heads, L, heads_dim)

        out = out.transpose(1, 2).contiguous().view(B, L, D)
        return self.W_o(out)


class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, hidden_dim):
        super().__init__()
        self.attn = AttentionBlock(embedding_dim, num_heads)
        self.ffn  = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embedding_dim),
        )
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        x = self.ln1(x + self.attn(x))
        x = self.ln2(x + self.ffn(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM)
        self.pos_emb = nn.Embedding(CONTEXT_LEN, EMBEDDING_DIM)
        self.blocks  = nn.Sequential(*[
            TransformerBlock(EMBEDDING_DIM, NUM_HEADS, HIDDEN_DIM)
            for _ in range(NUM_LAYERS)
        ])
        self.ln_f  = nn.LayerNorm(EMBEDDING_DIM)
        self.head  = nn.Linear(EMBEDDING_DIM, VOCAB_SIZE)

    def forward(self, idx, targets=None):
        B, L = idx.shape
        positions = torch.arange(L, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(positions)  # (B, L, D)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (B, L, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), targets.view(-1))

        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_crop = idx[:, -CONTEXT_LEN:]
            logits, _ = self(idx_crop)
            probs = torch.softmax(logits[:, -1, :], dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_tok], dim=1)
        return idx


# ---- training ----
model = GPT().to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(EVAL_STEPS):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

for step in range(STEPS):
    if step % EVAL_INTERVAL == 0:
        losses = estimate_loss()
        print(f"step {step:4d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f}")

    x, y = get_batch("train")
    _, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("\n--- sample ---")
ctx = torch.zeros((1, 1), dtype=torch.long, device=DEVICE)
print(decode(model.generate(ctx, max_new_tokens=500)[0].tolist()))

Vocab size: 65, dataset size: 1,115,394 chars
Parameters: 818,241
step    0 | train loss 4.3869 | val loss 4.3952
step  500 | train loss 2.1265 | val loss 2.1526
step 1000 | train loss 1.8756 | val loss 1.9545
step 1500 | train loss 1.7135 | val loss 1.8557
step 2000 | train loss 1.6213 | val loss 1.7857
step 2500 | train loss 1.5687 | val loss 1.7352
step 3000 | train loss 1.5120 | val loss 1.6929
step 3500 | train loss 1.4847 | val loss 1.6760
step 4000 | train loss 1.4495 | val loss 1.6530
step 4500 | train loss 1.4377 | val loss 1.6355

--- sample ---

Our whoe, spaccious us now you do met.

Post:
Green it, so said to ceft with him again!
Thou standly good, to lest me joy.

BRUTUS:
I ware.

CAPULET:
Away, do you seek poffections bat's hand seels
That most may Rome.

SLY:
To bany you; one eyeag that him and mine at showl:
I do bear thee thee to the ladys.

CAPULEW:
His conternets lebrange with not this
And strife! Trush this mrown cerrow
With his.
The will, but, every high uncorrise